In [ ]:
# 05 — Retrieval quality check for faq_index
# Asks FAQ-style questions (worded differently from the guide) and checks whether the chunk that
# answers each one is in the top K results. Each expected chunk is identified by a phrase that only
# appears in the answer, so the check survives re-chunking. Results are appended to rag_eval_runs.

In [ ]:
from databricks.sdk import WorkspaceClient

w          = WorkspaceClient()
catalog    = "bootcamp_students"
schema     = "maintops"
INDEX_NAME = f"{catalog}.{schema}.faq_index"
K          = 3
MIN_HIT_RATE = 0.8

CASES = [
    ("Can I pick which handyman does the job?",                 "you make the final decision"),
    ("Do I need to know if I need a plumber or an electrician?", "Do I need to know what kind of handyman I need?"),
    ("Is the closest handyman always chosen?",                  "not simply the closest handyman"),
    ("How much does it cost to use MaintOps?",                  "Pricing has not yet been defined"),
    ("Can I leave a review after the job is done?",             "give them a rating from 1 to 5"),
    ("I smell gas in my kitchen, what should I do?",            "suspected gas leak"),
    ("Which countries or cities is the service available in?",  "Germany and European users"),
    ("Can electricians sign up to get jobs?",                   "Handymen can create their own accounts"),
    ("Will the handyman arrive at a guaranteed time?",          "guaranteed arrival time"),
    ("What details should I give when reporting a problem?",    "Describe what you can observe"),
    ("How do you know if a handyman is too busy?",              "several active jobs"),
    ("What happens once I have chosen a handyman?",             "Once you select a handyman"),
]

In [ ]:
import json
from datetime import datetime, timezone

details, hits = [], 0
for question, phrase in CASES:
    res = w.api_client.do("POST", f"/api/2.0/vector-search/indexes/{INDEX_NAME}/query", body={
        "columns": ["chunk_id", "chunk_text"],
        "query_text": question,
        "num_results": K,
        "query_type": "hybrid",
    })
    rows = res["result"]["data_array"]
    rank = next((i + 1 for i, r in enumerate(rows) if phrase.lower() in r[1].lower()), None)
    hits += rank is not None
    details.append({"question": question, "expected_phrase": phrase, "rank": rank})
    print(f"{'HIT ' if rank else 'MISS'} rank={rank}  {question}")

hit_rate = hits / len(CASES)
print(f"\nhit@{K}: {hits}/{len(CASES)} = {hit_rate:.0%}")

In [ ]:
spark.createDataFrame([{
    "run_at": datetime.now(timezone.utc),
    "index_name": INDEX_NAME,
    "k": K,
    "n_questions": len(CASES),
    "hits": hits,
    "hit_rate": hit_rate,
    "details": json.dumps(details),
}]).write.mode("append").saveAsTable(f"{catalog}.{schema}.rag_eval_runs")

display(spark.table(f"{catalog}.{schema}.rag_eval_runs").orderBy("run_at", ascending=False).limit(5))

if hit_rate < MIN_HIT_RATE:
    raise RuntimeError(f"Retrieval quality below threshold: hit@{K} = {hit_rate:.0%} < {MIN_HIT_RATE:.0%}")